librerias

In [38]:
import networkx as nx
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GCNConv, GATConv
from torch_geometric.utils import from_networkx, negative_sampling
from torch_geometric.transforms import RandomLinkSplit
from sklearn.preprocessing import LabelEncoder
import os
import copy
import warnings
from funciones import (
    validar_par, 
    vectorize_node,
    eval_link_predictor
)

warnings.filterwarnings("ignore")

Probamos con 3 modelos: GraphSAGE ,GCN y GAT

In [39]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_channels[0]))
        for i in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_channels[i], hidden_channels[i+1]))
        self.convs.append(SAGEConv(hidden_channels[-1], out_channels))

    def encode(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = F.relu(conv(x, edge_index))
        return self.convs[-1](x, edge_index)

    def decode(self, z, edge_label_index):
        return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

    def project_new_node(self, x):
        for i, conv in enumerate(self.convs):
            x = conv.lin_l(x)
            if i < len(self.convs) - 1: x = F.relu(x)
        return x

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels[0]))
        for i in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels[i], hidden_channels[i+1]))
        self.convs.append(GCNConv(hidden_channels[-1], out_channels))

    def encode(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
        return self.convs[-1](x, edge_index)

    def decode(self, z, edge_label_index):
        return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

    def project_new_node(self, x):
        for i, conv in enumerate(self.convs):
            x = conv.lin(x)
            if i < len(self.convs) - 1: x = F.relu(x)
        return x

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, heads=2):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(GATConv(in_channels, hidden_channels[0], heads=heads, concat=True))
        for i in range(num_layers - 2):
            self.convs.append(GATConv(hidden_channels[i] * heads, hidden_channels[i+1], heads=heads, concat=True))
        self.convs.append(GATConv(hidden_channels[-1] * heads, out_channels, heads=1, concat=False))

    def encode(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.elu(x)
        return self.convs[-1](x, edge_index)

    def decode(self, z, edge_label_index):
        return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

    def project_new_node(self, x):
        for i, conv in enumerate(self.convs):
            x = conv.lin(x) 
            if i < len(self.convs) - 1: x = F.elu(x)
        return x

Creamos la funcion de entrenamiento para los modelos

In [40]:
def train_model_and_return_score(model, train_data, val_data, optimizer, criterion, epochs=300):
    best_val_auc = 0
    best_model_state = None
    final_loss = 0.0
    
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        
        z = model.encode(train_data.x, train_data.edge_index)
        
        neg_edge_index = negative_sampling(train_data.edge_index, num_nodes=train_data.num_nodes, 
                                         num_neg_samples=train_data.edge_label_index.size(1))
        edge_label_index = torch.cat([train_data.edge_label_index, neg_edge_index], dim=-1)
        edge_label = torch.cat([train_data.edge_label, torch.zeros(neg_edge_index.size(1))], dim=0)

        out = model.decode(z, edge_label_index).view(-1)
        loss = criterion(out, edge_label)
        loss.backward()
        optimizer.step()
        
        final_loss = loss.item()

        if epoch % 10 == 0:
            val_auc, _, _ = eval_link_predictor(model, val_data)
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_model_state = copy.deepcopy(model.state_dict())

    if best_model_state:
        model.load_state_dict(best_model_state)
        
    return model, best_val_auc, final_loss

Funcion para el proceso ejecucion para generar looks


In [ ]:
def generar_looks_finales(model, data, G, encoders, lista_entradas, output_csv="../Datos/Transformados/resultados_looks_final.csv"):
    
    # 1. Preparar modelo
    model.eval()
    with torch.no_grad():
        full_emb = model.encode(data.x, data.edge_index)
    
    datos_csv = []
    
    print(f"\nProcesando {len(lista_entradas)} prendas nuevas...")

    # 2. Iterar sobre cada prenda semilla
    for idx, seed in enumerate(lista_entradas):
        # Vectorizar y proyectar
        feat_vec = vectorize_node(seed, encoders)
        feat_tensor = torch.tensor([feat_vec], dtype=torch.float)
        
        with torch.no_grad():
            seed_emb = model.project_new_node(feat_tensor)
            scores = torch.matmul(full_emb, seed_emb.t()).sigmoid().view(-1)
        
        # 3. Obtener candidatos Top-K
        top_k = 600
        top_scores, top_idxs = torch.topk(scores, min(top_k, len(G.nodes)))
        
        candidates = {1: [], 2: [], 3: [], 4: []}
        
        for i, sc in zip(top_idxs.numpy(), top_scores.numpy()):
            node_id = list(G.nodes)[i]
            node_data = G.nodes[node_id]
            lvl = int(node_data.get('nivel', 0))
            
            # Filtramos por nivel y validamos compatibilidad básica con la semilla
            if lvl in candidates and validar_par(seed, node_data):
                candidates[lvl].append({'data': node_data, 'score': sc})

        # Datos básicos de la semilla
        seed_lvl = int(seed.get('nivel', 3))
        seed_name = seed.get('nombre_producto')
        
        # --- FUNCIÓN AUXILIAR PARA VARIEDAD ---
        def buscar_complemento(lista_candidatos, prenda_base, n_look_actual):
            """
            Busca un complemento compatible intentando no repetir siempre el primero.
            Usa n_look_actual para rotar el punto de inicio de la búsqueda.
            """
            total = len(lista_candidatos)
            if total == 0: return None
            
            # Rotación: Empezar a buscar en un índice diferente según el nº de look
            start_idx = n_look_actual % total
            
            # Reordenamos la lista: [C, D, A, B]
            lista_rotada = lista_candidatos[start_idx:] + lista_candidatos[:start_idx]
            
            for item in lista_rotada:
                candidate_data = item['data']
                if validar_par(candidate_data, prenda_base):
                    return candidate_data
            return None
        # ---------------------------------------

        # 4. Generar los 3 Looks
        for n_look in range(3):
            look_items = [seed]
            fila = {
                'Semilla_Producto': seed_name, 'Semilla_Nivel': seed_lvl, 'Look_ID': n_look + 1,
                'Nivel_1_Jumpsuit': '', 'Nivel_2_Bottom': '', 'Nivel_3_Top': '', 'Nivel_4_Outerwear': ''
            }
            
            # Colocar la semilla en su columna
            if seed_lvl == 1: fila['Nivel_1_Jumpsuit'] = seed_name
            elif seed_lvl == 2: fila['Nivel_2_Bottom'] = seed_name
            elif seed_lvl == 3: fila['Nivel_3_Top'] = seed_name
            elif seed_lvl == 4: fila['Nivel_4_Outerwear'] = seed_name

            # --- LÓGICA DE COMPLETADO (ACTUALIZADA) ---

            # CASO A: SEMILLA ES TOP (3)
            if seed_lvl == 3:
                pool_2 = candidates[2]
                if len(pool_2) > 0:
                    # Variedad: Elegir bottom basado en n_look
                    idx_bot = n_look % len(pool_2)
                    bot = pool_2[idx_bot]['data']
                    
                    fila['Nivel_2_Bottom'] = bot['nombre_producto']
                    look_items.append(bot)
                    
                    # Buscar Outerwear compatible con el Bottom elegido
                    out = buscar_complemento(candidates[4], bot, n_look)
                    if out: fila['Nivel_4_Outerwear'] = out['nombre_producto']

            # CASO B: SEMILLA ES BOTTOM (2)
            elif seed_lvl == 2:
                pool_3 = candidates[3]
                if len(pool_3) > 0:
                    # Variedad: Elegir Top basado en n_look
                    idx_top = n_look % len(pool_3)
                    top = pool_3[idx_top]['data']
                    
                    fila['Nivel_3_Top'] = top['nombre_producto']
                    look_items.append(top)
                    
                    # Buscar Outerwear compatible con el Top elegido
                    out = buscar_complemento(candidates[4], top, n_look)
                    if out: fila['Nivel_4_Outerwear'] = out['nombre_producto']

            # CASO C: SEMILLA ES OUTERWEAR (4)
            elif seed_lvl == 4:
                # Estrategia híbrida: Alternar entre Top+Bottom y Jumpsuit
                prefer_complex = (n_look % 2 == 0) # Looks 1 y 3 intentan ser complejos
                look_hecho = False

                if prefer_complex:
                    # Intento 1: Top + Bottom
                    if len(candidates[3]) > 0:
                        # Elegir Top rotando según n_look
                        idx_top = n_look % len(candidates[3])
                        top = candidates[3][idx_top]['data']
                        
                        # Buscar Bottom para este Top
                        bot = buscar_complemento(candidates[2], top, n_look)
                        
                        if bot:
                            fila['Nivel_3_Top'] = top['nombre_producto']
                            fila['Nivel_2_Bottom'] = bot['nombre_producto']
                            look_hecho = True
                
                # Si no era complejo o falló la búsqueda anterior, intentamos Jumpsuit
                if not look_hecho:
                    if len(candidates[1]) > 0:
                        # Elegir Jumpsuit rotando
                        idx_jump = n_look % len(candidates[1])
                        jump = candidates[1][idx_jump]['data']
                        fila['Nivel_1_Jumpsuit'] = jump['nombre_producto']
                        look_hecho = True
                    
                    # Fallback final: Si tampoco hay jumpsuit, forzar Top+Bottom con otra rotación
                    elif len(candidates[3]) > 0:
                        idx_top_fallback = (n_look + 1) % len(candidates[3])
                        top = candidates[3][idx_top_fallback]['data']
                        bot = buscar_complemento(candidates[2], top, n_look + 1)
                        if bot:
                            fila['Nivel_3_Top'] = top['nombre_producto']
                            fila['Nivel_2_Bottom'] = bot['nombre_producto']

            # CASO D: SEMILLA ES JUMPSUIT (1)
            elif seed_lvl == 1:
                pool_4 = candidates[4]
                if len(pool_4) > 0:
                    # Variedad: Elegir chaqueta rotando
                    idx_out = n_look % len(pool_4)
                    fila['Nivel_4_Outerwear'] = pool_4[idx_out]['data']['nombre_producto']

            datos_csv.append(fila)

    # 5. Exportar CSV
    if datos_csv:
        df_res = pd.DataFrame(datos_csv)
        cols = ['Semilla_Producto', 'Semilla_Nivel', 'Look_ID', 'Nivel_1_Jumpsuit', 'Nivel_2_Bottom', 'Nivel_3_Top', 'Nivel_4_Outerwear']
        df_res = df_res[cols]
        df_res.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"CSV generado: {output_csv}")
    else:
        print("No se generaron combinaciones.")

Cargar Modelo y hacer un split

In [42]:
path_graph = "../Modelos/graph.gml"
G = nx.read_gml(path_graph)

encoders = {k: LabelEncoder() for k in ['hexadecimal', 'categoria_prenda', 'weather', 'style']}
for attr in encoders:
    vals = [str(d.get(attr, 'unknown')) for _, d in G.nodes(data=True)]
    encoders[attr].fit(list(set(vals)) + ['unknown'])

for n in G.nodes(): 
    G.nodes[n]['x'] = vectorize_node(G.nodes[n], encoders)

data = from_networkx(G)
data.x = data.x.float()

train_data, val_data, _ = RandomLinkSplit(num_val=0.1, num_test=0, is_undirected=True)(data)

Entrenamiento de los 3 modelos

In [43]:
modelos_a_probar = ["SAGE", "GCN", "GAT"]
resultados = []
modelos_entrenados = {}

for nombre_modelo in modelos_a_probar:
    print(f"\n>>> Entrenando modelo: {nombre_modelo}...")
    
    if nombre_modelo == "SAGE":
        model = GraphSAGE(data.x.shape[1], [64, 32], 32, 3)
        lr = 0.005
    elif nombre_modelo == "GCN":
        model = GCN(data.x.shape[1], [64, 32], 32, 3)
        lr = 0.01
    elif nombre_modelo == "GAT":
        model = GAT(data.x.shape[1], [32, 16], 32, 3, heads=2) 
        lr = 0.005

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.BCEWithLogitsLoss()

    model_trained, best_auc, last_loss = train_model_and_return_score(
        model, train_data, val_data, optimizer, criterion, epochs=500
    )
    
    resultados.append({
        "Modelo": nombre_modelo,
        "Best AUC": best_auc,
        "Final Loss": last_loss
    })
    modelos_entrenados[nombre_modelo] = model_trained
    print(f"    Terminado: AUC={best_auc:.4f}, Loss={last_loss:.4f}")


>>> Entrenando modelo: SAGE...
    Terminado: AUC=0.6937, Loss=0.6747

>>> Entrenando modelo: GCN...
    Terminado: AUC=0.6991, Loss=0.6657

>>> Entrenando modelo: GAT...
    Terminado: AUC=0.7518, Loss=0.6681


Gana en base al AUC

In [44]:

df_resultados = pd.DataFrame(resultados)

print(df_resultados)

# Seleccionar automáticamente el mejor por AUC
mejor_modelo_row = df_resultados.loc[df_resultados['Best AUC'].idxmax()]
BEST_MODEL_NAME = mejor_modelo_row['Modelo']

print(f"El mejor modelo es: {BEST_MODEL_NAME}")

final_model = modelos_entrenados[BEST_MODEL_NAME]

  Modelo  Best AUC  Final Loss
0   SAGE  0.693650    0.674662
1    GCN  0.699070    0.665659
2    GAT  0.751791    0.668051
El mejor modelo es: GAT


Hacemos una busqueda de hiperparametros para el mejor modelo (GAT)

In [45]:
import itertools
import copy


param_grid = {
    'hidden_channels': [32, 64],    
    'heads': [4],              
    'lr': [0.005]           
}

def grid_search_gat(data, train_data, val_data, grid):
    results = []
    best_overall_auc = 0
    best_model_overall = None
    best_params = {}

    keys, values = zip(*grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    print(f"Grid Search con {len(combinations)} combinaciones...\n")

    for i, params in enumerate(combinations):
        print(f"--- Combinación {i+1}/{len(combinations)}: {params} ---")
        
        model = GAT(
            in_channels=data.x.shape[1],
            hidden_channels=[params['hidden_channels'], params['hidden_channels'] // 2], 
            out_channels=32,
            num_layers=3,
            heads=params['heads']
        )
        
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
        criterion = torch.nn.BCEWithLogitsLoss()

        model_trained, best_val_auc, final_loss = train_model_and_return_score(
            model, train_data, val_data, optimizer, criterion, epochs=200
        )
        
        print(f"    Resultado: AUC={best_val_auc:.4f} | Loss={final_loss:.4f}")
        
        results.append({
            **params,
            'auc': best_val_auc,
            'loss': final_loss
        })

        if best_val_auc > best_overall_auc:
            best_overall_auc = best_val_auc
            best_model_overall = copy.deepcopy(model_trained)
            best_params = params
            print("------------------------------")
            print("¡Nuevo MEJOR modelo encontrado!")
            print("------------------------------")


    return best_model_overall, best_params, pd.DataFrame(results)



best_model_, best_params, df_results = grid_search_gat(data, train_data, val_data, param_grid)


Grid Search con 2 combinaciones...

--- Combinación 1/2: {'hidden_channels': 32, 'heads': 4, 'lr': 0.005} ---


    Resultado: AUC=0.7468 | Loss=0.6687
------------------------------
¡Nuevo MEJOR modelo encontrado!
------------------------------
--- Combinación 2/2: {'hidden_channels': 64, 'heads': 4, 'lr': 0.005} ---
    Resultado: AUC=0.7957 | Loss=0.6654
------------------------------
¡Nuevo MEJOR modelo encontrado!
------------------------------


Aqui metemos 10 prendas nuevas que el grafo no conoce

In [46]:
NUEVAS_PRENDAS_ENTRADA = [
    {'nombre_producto': 'Onljade Cardigan knit Black', 'hexadecimal': '000000', 'categoria_prenda': 'top', 'nivel': 3, 'style': 'boho', 'weather': 'cold_season'},
    {'nombre_producto': 'Onljade Cardigan knit Beige', 'hexadecimal': 'DFC8B2', 'categoria_prenda': 'top', 'nivel': 3, 'style': 'boho', 'weather': 'cold_season'},
    {'nombre_producto': 'Onljade Cardigan knit Brown', 'hexadecimal': 'B94600', 'categoria_prenda': 'top', 'nivel': 3, 'style': 'boho', 'weather': 'cold_season'},
    {'nombre_producto': 'Cerise Jacket print Blue', 'hexadecimal': '00008B', 'categoria_prenda': 'outerwear', 'nivel': 4, 'style': 'boho', 'weather': 'warm_season'},
    {'nombre_producto': 'Cerise Jacket print Black', 'hexadecimal': '000000', 'categoria_prenda': 'outerwear', 'nivel': 4, 'style': 'boho', 'weather': 'warm_season'},
    {'nombre_producto': 'Combi Jumpsuit long', 'hexadecimal': '000000', 'categoria_prenda': 'jumpsuit', 'nivel': 1, 'style': 'classic', 'weather': 'cold_season'},
    {'nombre_producto': 'Ante Pant', 'hexadecimal': '000000', 'categoria_prenda': 'bottoms', 'nivel': 2, 'style': 'street', 'weather': 'cold'},
    {'nombre_producto': 'Dena Pant jegging', 'hexadecimal': '000000', 'categoria_prenda': 'bottoms', 'nivel': 2, 'style': 'street', 'weather': 'cold_season'},
    {'nombre_producto': 'Pcjenna Scarf long Grey', 'hexadecimal': '5F5E5E', 'categoria_prenda': 'scarf', 'nivel': 4, 'style': 'casual', 'weather': 'cold'},
    {'nombre_producto': 'Pcjenna Scarf long Orange', 'hexadecimal': 'FFA500', 'categoria_prenda': 'scarf', 'nivel': 4, 'style': 'casual', 'weather': 'cold'},
    {'nombre_producto': 'Combi Jumpsuit long', 'hexadecimal': '000000', 'categoria_prenda': 'jumpsuit', 'nivel': 3, 'style': 'classic', 'weather': 'cold_season'},
    {'nombre_producto': 'Ante Pant', 'hexadecimal': '000000', 'categoria_prenda': 'bottoms', 'nivel': 3, 'style': 'street', 'weather': 'cold'}
]
generar_looks_finales(best_model_, data, G, encoders, NUEVAS_PRENDAS_ENTRADA, output_csv="../Datos/Transformados/resultados_looks_optimizado.csv")


Procesando 12 prendas nuevas...
✅ CSV generado correctamente con variedad mejorada: ../Datos/Transformados/resultados_looks_optimizado.csv


mejores parametros para GAT

In [47]:
best_params

{'hidden_channels': 64, 'heads': 4, 'lr': 0.005}

guardamos el modelo en .pth

In [48]:
def guardar_para_produccion(model, encoders, params, path="../Modelos/Modelo_GAT.pth"):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'encoders': encoders,
        'model_config': params,  
        'input_dim': data.x.shape[1]
    }
    torch.save(checkpoint, path)

guardar_para_produccion(best_model_, encoders, best_params)

In [49]:
import pandas as pd

In [50]:
df=pd.read_csv("../Datos/Transformados/resultados_looks_optimizado.csv")
df

,Semilla_Producto,Semilla_Nivel,Look_ID,Nivel_1_Jumpsuit,Nivel_2_Bottom,Nivel_3_Top,Nivel_4_Outerwear
0,Onljade Cardigan knit Black,3,1,NaN,Saturne Pant grace,Onljade Cardigan knit Black,Viblue Jacket noos
1,Onljade Cardigan knit Black,3,2,NaN,Alexa Jeans slim,Onljade Cardigan knit Black,Viblue Jacket noos
2,Onljade Cardigan knit Black,3,3,NaN,Adina Jeans button,Onljade Cardigan knit Black,Visti Jacket bomber
3,Onljade Cardigan knit Beige,3,1,NaN,Dana Skirt pencil,Onljade Cardigan knit Beige,Gold Jacket bomb
4,Onljade Cardigan knit Beige,3,2,NaN,Vicom Jeans noos,Onljade Cardigan knit Beige,Laura Jacket jo
5,Onljade Cardigan knit Beige,3,3,NaN,Freeman Jeans mavuto,Onljade Cardigan knit Beige,Vmcallie Jacket bomber
6,Onljade Cardigan knit Brown,3,1,NaN,Dana Skirt pencil,Onljade Cardigan knit Brown,Gold Jacket bomb
7,Onljade Cardigan knit Brown,3,2,NaN,Vicom Jeans noos,Onljade Cardigan knit Brown,Laura Jacket jo
8,Onljade Cardigan knit Brown,3,3,NaN,Freeman Jeans mavuto,Onljade Cardigan knit Brown,Viblue Jacket noos
9,Cerise Jacket print Blue,4,1,NaN,Senes Short stripes,Medos Tshirt slub,Cerise Jacket print Blue
